# Exploratory Data Analysis (EDA) on the Wine Dataset
### Tools: NumPy · Pandas · Matplotlib · Seaborn · Scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded")

## 2. Load the Wine Dataset
Scikit-learn ships a classic **Wine** dataset (178 samples, 13 chemical features, 3 wine cultivars).

In [ ]:
wine = load_wine()

df = pd.DataFrame(wine.data, columns=wine.feature_names)

df["target"] = wine.target
df["cultivar"] = df["target"].map({0: "class_0", 1: "class_1", 2: "class_2"})

print("Shape of dataset:", df.shape)
df.head(10)

In [ ]:
print("Rows, Columns:", df.shape)
print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Info ---")
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(df["cultivar"].value_counts())

## 3. Univariate Analysis
Analysis **one variable at a time** — its distribution, spread, and outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="cultivar", ax=axes[0], palette="Set2")
axes[0].set_title("Count of Each Wine Cultivar")
axes[0].set_xlabel("Cultivar")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())}",
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom", fontsize=11, fontweight="bold")


counts = df["cultivar"].value_counts()
axes[1].pie(counts, labels=counts.index, autopct="%1.1f%%",
            startangle=90, colors=sns.color_palette("Set2"))
axes[1].set_title("Proportion of Cultivars")

plt.tight_layout()
plt.show()

In [ ]:
df.drop(columns=["target", "cultivar"]).hist(
    figsize=(16, 12),
    bins=20,
    color="#69b3a2",
    edgecolor="black"
)
plt.suptitle("Histogram of All Features", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = ["alcohol", "malic_acid", "ash", "flavanoids"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(features_to_plot):
    sns.kdeplot(data=df, x=col, hue="cultivar",
                fill=True, ax=axes[i], palette="Set2", alpha=0.5)
    axes[i].set_title(f"KDE of {col}")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
features = ['alcohol', 'malic_acid', 'flavanoids']

for i, col in enumerate(features):
    sns.boxplot(data=df, y=col, ax=axes[i], palette="Set2")
    axes[i].set_title(f"Boxplot of {col.capitalize()}") 

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
features_violin_plots = ['alcohol', 'flavanoids', 'color_intensity']

for i, col in enumerate(features_violin_plots):
    sns.violinplot(data=df, x="cultivar", y=col, ax=axes[i], palette="Set2", inner="quart")
    axes[i].set_title(f"{col.capitalize()} by Cultivar")

plt.tight_layout()
plt.show()

## 4. Bivariate Analysis
examine **relationships between two variables**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
features_scatter_plots = [{'x': 'alcohol', 'y': 'flavanoids'}, {'x': 'color_intensity', 'y': 'malic_acid'}, {'x': 'proline', 'y': 'alcohol'}]

for i, hv in enumerate(features_scatter_plots):
    sns.scatterplot(data=df, x=hv['x'], y=hv['y'],
                    hue="cultivar", palette="Set2", s=80, ax=axes[i])
    axes[i].set_title(f"{hv['x'].capitalize()} vs {hv['y'].capitalize()}")

plt.tight_layout()
plt.show()

In [ ]:
sns.jointplot(data=df, x="flavanoids", y="od280/od315_of_diluted_wines",
              hue="cultivar", palette="Set1", height=7, alpha=0.7)
plt.suptitle("Joint Plot: Flavanoids vs OD280/OD315", y=1.02, fontweight="bold")
plt.show()

In [ ]:
plt.figure(figsize=(13, 10))
corr = df.drop(columns=["target", "cultivar"]).corr()

sns.heatmap(corr,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            center=0,
            linewidths=0.5,
            square=True,
            cbar_kws={"shrink": .8})
plt.title("Correlation Matrix of Wine Features")
plt.tight_layout()
plt.show()

In [ ]:
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1].drop_duplicates().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=corr_pairs.values, y=[f"{a} <-> {b}" for a, b in corr_pairs.index],
            palette="viridis")
plt.title("Top 10 Strongest Correlated Feature Pairs")
plt.xlabel("Absolute Correlation")
plt.tight_layout()
plt.show()

In [ ]:
selected = ["alcohol", "flavanoids", "color_intensity", "proline", "cultivar"]
sns.pairplot(df[selected], hue="cultivar", palette="Set2",
             diag_kind="kde", corner=True)
plt.suptitle("Pairplot of Selected Features", y=1.02, fontweight="bold")
plt.show()

In [ ]:
features = ["alcohol", "malic_acid", "ash", "alcalinity_of_ash",
            "magnesium", "total_phenols"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.boxplot(data=df, x="cultivar", y=col, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} by Cultivar")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.stripplot(data=df, x="cultivar", y="flavanoids",
              palette="Set2", jitter=True, size=6, ax=axes[0])
axes[0].set_title("Strip Plot: Flavanoids by Cultivar")

sns.swarmplot(data=df, x="cultivar", y="flavanoids",
              palette="Set2", size=6, ax=axes[1])
axes[1].set_title("Swarm Plot: Flavanoids by Cultivar")

plt.tight_layout()
plt.show()

## 5. Outlier Detection with IQR

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.drop(["target"])

outlier_summary = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary[col] = len(outliers)

outlier_df = pd.DataFrame(outlier_summary.items(),
                          columns=["Feature", "Outlier_Count"]).sort_values(
                          by="Outlier_Count", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=outlier_df, x="Outlier_Count", y="Feature", palette="rocket")
plt.title("Number of Outliers per Feature (IQR Method)")
plt.tight_layout()
plt.show()

outlier_df

## 6. Feature Scaling
StandardScaler makes features comparable (mean = 0, std = 1).

In [ ]:
X = df.drop(columns=["target", "cultivar"])
y = df["target"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Before vs After
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=X, ax=axes[0], palette="Set3")
axes[0].set_title("Before Scaling")
axes[0].tick_params(axis="x", rotation=90)

sns.boxplot(data=X_scaled_df, ax=axes[1], palette="Set3")
axes[1].set_title("After Standard Scaling")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()